# Notebook 02 — Function Calling y Tool Use

**Semana:** 2  
**Bloque:** 3.2 — Function calling / tool use  
**Tiempo estimado:** 2.5-3.5 horas

## Qué vas a construir

Al terminar este notebook tendrás:
- El loop completo de tool use implementado para OpenAI y Anthropic
- Un asistente con tres herramientas reales funcionando
- Parallel tool use: el modelo llama dos herramientas a la vez
- Un registro de todas las herramientas invocadas por sesión
- Un mini-framework reutilizable para registrar y ejecutar herramientas

## Prerrequisitos

- Notebook 01 completado (clientes OpenAI y Anthropic inicializados)
- `pip install openai anthropic python-dotenv requests wikipedia-api`

---
## Celda 0 — Setup

In [ ]:
import os
import json
import math
import datetime
import requests
from dotenv import load_dotenv
from openai import OpenAI
import anthropic

load_dotenv()

cliente_openai    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
cliente_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("Clientes inicializados correctamente.")

---
## Celda 1 — Implementa las tres herramientas reales

Antes de ocuparnos del LLM, implementa las funciones Python que serán las herramientas.
Son funciones normales — el LLM no las toca, solo las llamas tú.

Las tres herramientas:
1. `obtener_fecha_hora` — devuelve la fecha y hora actual
2. `calcular` — evalúa expresiones matemáticas de forma segura
3. `buscar_wikipedia` — devuelve el resumen de un artículo de Wikipedia

In [ ]:
# ─── Herramienta 1: fecha y hora actual ───────────────────────────────────────

def obtener_fecha_hora(zona_horaria: str = "Europe/Madrid") -> dict:
    """
    Devuelve la fecha y hora actual.
    Nota: en producción usarías pytz o zoneinfo para zonas horarias reales.
    Aquí simplificamos con UTC.
    """
    ahora = datetime.datetime.utcnow()
    return {
        "fecha":         ahora.strftime("%Y-%m-%d"),
        "hora":          ahora.strftime("%H:%M:%S"),
        "dia_semana":    ahora.strftime("%A"),
        "zona_horaria":  "UTC (aproximado)",
        "timestamp_iso": ahora.isoformat()
    }


# ─── Herramienta 2: calculadora segura ────────────────────────────────────────

def calcular(expresion: str) -> dict:
    """
    Evalúa una expresión matemática de forma segura.
    Permite: +, -, *, /, **, %, sqrt, sin, cos, tan, log, pi, e
    """
    # Whitelist de funciones permitidas — NUNCA uses eval() sin restricciones
    funciones_permitidas = {
        "sqrt":  math.sqrt,
        "sin":   math.sin,
        "cos":   math.cos,
        "tan":   math.tan,
        "log":   math.log,
        "log10": math.log10,
        "pi":    math.pi,
        "e":     math.e,
        "abs":   abs,
        "round": round,
    }
    try:
        resultado = eval(expresion, {"__builtins__": {}}, funciones_permitidas)
        return {
            "expresion": expresion,
            "resultado": resultado,
            "error":     None
        }
    except Exception as e:
        return {
            "expresion": expresion,
            "resultado": None,
            "error":     str(e)
        }


# ─── Herramienta 3: búsqueda en Wikipedia ────────────────────────────────────

def buscar_wikipedia(tema: str, idioma: str = "es", oraciones: int = 3) -> dict:
    """
    Devuelve el resumen de un artículo de Wikipedia.
    Usa la API REST de Wikipedia (no necesita clave).
    """
    url = f"https://{idioma}.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(tema)}"
    try:
        respuesta = requests.get(url, timeout=5)
        respuesta.raise_for_status()
        datos = respuesta.json()
        
        # Recorta a N oraciones
        extracto = datos.get("extract", "")
        oraciones_lista = extracto.split(". ")
        resumen = ". ".join(oraciones_lista[:oraciones]) + "."
        
        return {
            "tema":     tema,
            "titulo":   datos.get("title", tema),
            "resumen":  resumen,
            "url":      datos.get("content_urls", {}).get("desktop", {}).get("page", ""),
            "error":    None
        }
    except requests.exceptions.HTTPError:
        return {
            "tema":    tema,
            "resumen": None,
            "error":   f"Artículo '{tema}' no encontrado en Wikipedia ({idioma})."
        }
    except Exception as e:
        return {
            "tema":    tema,
            "resumen": None,
            "error":   str(e)
        }


# ─── Prueba manual de las tres herramientas ──────────────────────────────────
print("=== Herramienta 1: fecha y hora ===")
print(json.dumps(obtener_fecha_hora(), indent=2, ensure_ascii=False))

print("\n=== Herramienta 2: calculadora ===")
print(json.dumps(calcular("sqrt(144) + pi * 2"), indent=2))
print(json.dumps(calcular("1 / 0"), indent=2))  # caso de error

print("\n=== Herramienta 3: Wikipedia ===")
print(json.dumps(buscar_wikipedia("Inteligencia artificial"), indent=2, ensure_ascii=False))

---
## Celda 2 — Define las herramientas en formato JSON Schema

Ahora describes las mismas herramientas en el formato que entiende el LLM.
Recuerda: la `description` es lo que el modelo lee para decidir cuándo usarlas.

In [ ]:
# ─── Definiciones para OpenAI ─────────────────────────────────────────────────

HERRAMIENTAS_OPENAI = [
    {
        "type": "function",
        "function": {
            "name": "obtener_fecha_hora",
            "description": (
                "Devuelve la fecha y hora actuales. "
                "Úsala cuando el usuario pregunte qué día es hoy, qué hora es, "
                "el día de la semana, o cualquier referencia al momento presente. "
                "NO la uses para fechas históricas o futuras."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "zona_horaria": {
                        "type": "string",
                        "description": "Zona horaria en formato IANA, p.ej. 'Europe/Madrid'. Por defecto: Europe/Madrid."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calcular",
            "description": (
                "Evalúa expresiones matemáticas. "
                "Úsala para cualquier cálculo numérico: aritmética, raíces, trigonometría, logaritmos. "
                "La expresión debe estar en Python matemático, p.ej. 'sqrt(25) + 3*4'. "
                "NO la uses para preguntas conceptuales sobre matemáticas."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expresion": {
                        "type": "string",
                        "description": "Expresión matemática válida en Python, p.ej. '2**10' o 'sqrt(144)'"
                    }
                },
                "required": ["expresion"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_wikipedia",
            "description": (
                "Busca y devuelve el resumen de un artículo de Wikipedia. "
                "Úsala cuando necesites información enciclopédica sobre personas, lugares, "
                "conceptos, eventos históricos o científicos. "
                "NO la uses para eventos muy recientes (Wikipedia puede no estar actualizada) "
                "ni para opiniones o recomendaciones."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tema": {
                        "type": "string",
                        "description": "Término de búsqueda tal como aparecería en el título del artículo de Wikipedia"
                    },
                    "idioma": {
                        "type": "string",
                        "enum": ["es", "en", "fr", "de", "it", "pt"],
                        "description": "Código de idioma de Wikipedia. Por defecto: 'es' (español)"
                    },
                    "oraciones": {
                        "type": "integer",
                        "description": "Número de oraciones del resumen. Entre 1 y 5. Por defecto: 3."
                    }
                },
                "required": ["tema"]
            }
        }
    }
]


# ─── Definiciones para Anthropic ─────────────────────────────────────────────
# La lógica es idéntica; solo cambia la estructura del JSON.

HERRAMIENTAS_ANTHROPIC = [
    {
        "name": "obtener_fecha_hora",
        "description": (
            "Devuelve la fecha y hora actuales. "
            "Úsala cuando el usuario pregunte qué día es hoy, qué hora es, "
            "el día de la semana, o cualquier referencia al momento presente."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "zona_horaria": {
                    "type": "string",
                    "description": "Zona horaria en formato IANA. Por defecto: Europe/Madrid."
                }
            },
            "required": []
        }
    },
    {
        "name": "calcular",
        "description": (
            "Evalúa expresiones matemáticas. Úsala para cualquier cálculo numérico. "
            "La expresión debe estar en Python matemático."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expresion": {
                    "type": "string",
                    "description": "Expresión matemática válida en Python"
                }
            },
            "required": ["expresion"]
        }
    },
    {
        "name": "buscar_wikipedia",
        "description": (
            "Busca y devuelve el resumen de un artículo de Wikipedia. "
            "Úsala para información enciclopédica sobre personas, lugares o conceptos."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "tema": {
                    "type": "string",
                    "description": "Término de búsqueda"
                },
                "idioma": {
                    "type": "string",
                    "enum": ["es", "en", "fr", "de", "it", "pt"],
                    "description": "Código de idioma. Por defecto: 'es'"
                },
                "oraciones": {
                    "type": "integer",
                    "description": "Número de oraciones del resumen (1-5). Por defecto: 3."
                }
            },
            "required": ["tema"]
        }
    }
]


# ─── Dispatcher: mapea nombre de herramienta → función Python ─────────────────

FUNCIONES_DISPONIBLES = {
    "obtener_fecha_hora": obtener_fecha_hora,
    "calcular":           calcular,
    "buscar_wikipedia":   buscar_wikipedia,
}

def ejecutar_herramienta(nombre: str, argumentos: dict) -> str:
    """Ejecuta la herramienta por nombre y devuelve el resultado como JSON string."""
    if nombre not in FUNCIONES_DISPONIBLES:
        return json.dumps({"error": f"Herramienta '{nombre}' no encontrada."})
    
    funcion  = FUNCIONES_DISPONIBLES[nombre]
    resultado = funcion(**argumentos)
    return json.dumps(resultado, ensure_ascii=False)


print(f"Herramientas registradas: {list(FUNCIONES_DISPONIBLES.keys())}")

---
## Celda 3 — Loop completo de tool use con OpenAI

Implementa el loop que:
1. Llama al modelo con las herramientas disponibles
2. Si el modelo pide herramientas, las ejecuta
3. Devuelve los resultados al modelo
4. Repite hasta que el modelo genere una respuesta de texto final

Estudia el flujo antes de ejecutar. ¿Cuántas llamadas HTTP se hacen para una pregunta que usa una herramienta?

In [ ]:
def asistente_con_herramientas_openai(
    pregunta: str,
    system_prompt: str = "Eres un asistente útil con acceso a herramientas. Responde siempre en español.",
    verbose: bool = True
) -> str:
    """
    Ejecuta el loop completo de tool use con OpenAI.
    Devuelve la respuesta final en texto.
    """
    historial = [
        {"role": "system",  "content": system_prompt},
        {"role": "user",    "content": pregunta}
    ]
    
    iteracion = 0
    max_iteraciones = 5  # evitar loops infinitos
    
    while iteracion < max_iteraciones:
        iteracion += 1
        if verbose:
            print(f"\n[Iteración {iteracion}] Llamando al modelo...")
        
        respuesta = cliente_openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=historial,
            tools=HERRAMIENTAS_OPENAI,
            tool_choice="auto"
        )
        
        mensaje = respuesta.choices[0].message
        
        # ── Caso A: el modelo quiere usar herramientas ──────────────────────
        if mensaje.tool_calls:
            # Añade la respuesta del modelo al historial
            historial.append(mensaje)
            
            # Ejecuta cada herramienta y añade el resultado al historial
            for tool_call in mensaje.tool_calls:
                nombre    = tool_call.function.name
                argumentos = json.loads(tool_call.function.arguments)
                
                if verbose:
                    print(f"  → Ejecutando: {nombre}({argumentos})")
                
                resultado = ejecutar_herramienta(nombre, argumentos)
                
                if verbose:
                    print(f"  ← Resultado: {resultado}")
                
                historial.append({
                    "role":         "tool",
                    "tool_call_id": tool_call.id,
                    "content":      resultado
                })
            
            # Continúa el loop: el modelo ahora tiene los resultados
            continue
        
        # ── Caso B: el modelo devuelve una respuesta de texto → fin del loop ─
        texto_final = mensaje.content
        if verbose:
            print(f"\n[Respuesta final tras {iteracion} iteración(es)]")
        return texto_final
    
    return "[Error: se alcanzó el máximo de iteraciones sin respuesta final]"


# ─── Pruebas ─────────────────────────────────────────────────────────────────

# Prueba 1: pregunta que necesita la herramienta de fecha
print("\n" + "="*60)
print("PREGUNTA 1: fecha y hora")
print("="*60)
respuesta1 = asistente_con_herramientas_openai("¿Qué día de la semana es hoy?")
print(f"\nRespuesta: {respuesta1}")

# Prueba 2: pregunta que necesita la calculadora
print("\n" + "="*60)
print("PREGUNTA 2: cálculo")
print("="*60)
respuesta2 = asistente_con_herramientas_openai("¿Cuánto es la raíz cuadrada de 17161?")
print(f"\nRespuesta: {respuesta2}")

# Prueba 3: pregunta que NO necesita herramientas
print("\n" + "="*60)
print("PREGUNTA 3: sin herramientas")
print("="*60)
respuesta3 = asistente_con_herramientas_openai("¿Qué es un transformer en IA?")
print(f"\nRespuesta: {respuesta3}")

---
## Celda 4 — Loop completo con Anthropic

El mismo loop, adaptado a las diferencias de la API de Anthropic.
Compara este código con el de la celda anterior: las diferencias son exactamente
las que documentaste en la dispensa.

In [ ]:
def asistente_con_herramientas_anthropic(
    pregunta: str,
    system_prompt: str = "Eres un asistente útil con acceso a herramientas. Responde siempre en español.",
    verbose: bool = True
) -> str:
    """
    Ejecuta el loop completo de tool use con Anthropic.
    """
    historial   = [{"role": "user", "content": pregunta}]
    iteracion   = 0
    max_iteraciones = 5
    
    while iteracion < max_iteraciones:
        iteracion += 1
        if verbose:
            print(f"\n[Iteración {iteracion}] Llamando al modelo...")
        
        respuesta = cliente_anthropic.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=system_prompt,
            tools=HERRAMIENTAS_ANTHROPIC,
            messages=historial
        )
        
        # ── Caso A: el modelo quiere usar herramientas ──────────────────────
        # En Anthropic: stop_reason == "tool_use" y hay bloques tool_use en content
        if respuesta.stop_reason == "tool_use":
            
            # Añade la respuesta del modelo al historial
            historial.append({
                "role":    "assistant",
                "content": respuesta.content  # lista de bloques (text + tool_use)
            })
            
            # Recoge los resultados de todas las herramientas
            resultados_herramientas = []
            
            for bloque in respuesta.content:
                if bloque.type == "tool_use":
                    nombre     = bloque.name
                    argumentos = bloque.input
                    
                    if verbose:
                        print(f"  → Ejecutando: {nombre}({argumentos})")
                    
                    resultado = ejecutar_herramienta(nombre, argumentos)
                    
                    if verbose:
                        print(f"  ← Resultado: {resultado}")
                    
                    resultados_herramientas.append({
                        "type":        "tool_result",
                        "tool_use_id": bloque.id,
                        "content":     resultado
                    })
            
            # Los resultados van en un único mensaje de rol "user"
            historial.append({
                "role":    "user",
                "content": resultados_herramientas
            })
            continue
        
        # ── Caso B: respuesta de texto final ────────────────────────────────
        texto_final = "".join(
            bloque.text for bloque in respuesta.content
            if hasattr(bloque, "text")
        )
        if verbose:
            print(f"\n[Respuesta final tras {iteracion} iteración(es)]")
        return texto_final
    
    return "[Error: se alcanzó el máximo de iteraciones sin respuesta final]"


# ─── Mismas pruebas que con OpenAI — compara los resultados ──────────────────

print("\n" + "="*60)
print("PREGUNTA 1: fecha y hora (Anthropic)")
print("="*60)
respuesta1 = asistente_con_herramientas_anthropic("¿Qué día de la semana es hoy?")
print(f"\nRespuesta: {respuesta1}")

print("\n" + "="*60)
print("PREGUNTA 2: Wikipedia (Anthropic)")
print("="*60)
respuesta2 = asistente_con_herramientas_anthropic("¿Quién fue Alan Turing y qué aportó a la informática?")
print(f"\nRespuesta: {respuesta2}")

---
## Celda 5 — Parallel tool use

Lanza una pregunta que requiere dos herramientas independientes a la vez.
Observa que el modelo genera DOS tool_calls en una sola respuesta,
y tú ejecutas ambas antes de volver a llamar al modelo.

In [ ]:
PREGUNTA_PARALELA = (
    "Necesito dos cosas: primero, dime cuántos días faltan para el 31 de diciembre "
    "(usa la herramienta de fecha para saber la fecha de hoy y calcula con la calculadora). "
    "Segundo, dime quién fue Marie Curie en dos frases."
)

print("Pregunta:", PREGUNTA_PARALELA)
print("\nEjecutando con OpenAI (verbose=True para ver el parallel tool use):")
print("="*60)

respuesta_paralela = asistente_con_herramientas_openai(PREGUNTA_PARALELA, verbose=True)
print(f"\nRespuesta final:\n{respuesta_paralela}")

**Pregunta de reflexión:** ¿cuántas iteraciones del loop esperabas? ¿Cuántas ocurrieron realmente? ¿Por qué?

Tu respuesta: ...

---
## Reto final — Mini-framework con registro de uso de herramientas

Construye una versión mejorada del asistente que:
1. Registra un log de cada herramienta invocada (nombre, argumentos, resultado, timestamp)
2. Al final de la sesión, imprime un resumen: cuántas veces se usó cada herramienta
3. Funciona con ambos proveedores (openai / anthropic) mediante un parámetro

Lee el esqueleto, completa los `# TODO` y pruébalo con al menos 5 preguntas variadas.

In [ ]:
from collections import defaultdict
from typing import Literal

class AsistenteConHerramientas:
    """
    Asistente multi-proveedor con registro de uso de herramientas.
    """
    
    def __init__(
        self,
        proveedor: Literal["openai", "anthropic"] = "openai",
        system_prompt: str = "Eres un asistente útil. Responde siempre en español."
    ):
        self.proveedor    = proveedor
        self.system_prompt = system_prompt
        self.log_herramientas: list[dict] = []       # registro completo
        self.contador     = defaultdict(int)          # nombre → nº de veces usada
    
    def _ejecutar_y_registrar(self, nombre: str, argumentos: dict) -> str:
        """Ejecuta la herramienta y registra la invocación."""
        resultado = ejecutar_herramienta(nombre, argumentos)
        
        self.log_herramientas.append({
            "timestamp":  datetime.datetime.utcnow().isoformat(),
            "herramienta": nombre,
            "argumentos": argumentos,
            "resultado":  resultado
        })
        self.contador[nombre] += 1
        
        return resultado
    
    def preguntar(self, pregunta: str, verbose: bool = False) -> str:
        """Envía una pregunta y devuelve la respuesta final."""
        # TODO: implementa el loop completo aquí.
        # Usa self.proveedor para decidir qué cliente y qué herramientas usar.
        # Usa self._ejecutar_y_registrar() en lugar de ejecutar_herramienta() directamente.
        #
        # Tip: puedes reutilizar casi todo el código de las celdas 3 y 4,
        # pero sustituyendo ejecutar_herramienta() por self._ejecutar_y_registrar()
        
        raise NotImplementedError("Implementa este método")
    
    def resumen_uso(self) -> None:
        """Imprime el resumen de uso de herramientas de la sesión."""
        print(f"\n{'='*40}")
        print("RESUMEN DE USO DE HERRAMIENTAS")
        print(f"{'='*40}")
        print(f"Total de invocaciones: {sum(self.contador.values())}")
        for nombre, veces in sorted(self.contador.items(), key=lambda x: -x[1]):
            print(f"  {nombre}: {veces} vez/veces")
        print(f"{'='*40}")
    
    def exportar_log(self, ruta: str = "herramientas_log.json") -> None:
        """Guarda el log completo en un archivo JSON."""
        with open(ruta, "w", encoding="utf-8") as f:
            json.dump(self.log_herramientas, f, indent=2, ensure_ascii=False)
        print(f"Log guardado en {ruta} ({len(self.log_herramientas)} entradas)")


# ─── Prueba del mini-framework ────────────────────────────────────────────────
# Descomenta cuando hayas implementado el método preguntar()

# asistente = AsistenteConHerramientas(proveedor="openai")

# preguntas = [
#     "¿Qué hora es ahora mismo?",
#     "¿Cuánto es 2 elevado a 32?",
#     "Dime algo sobre la Revolución Francesa.",
#     "¿Qué día es hoy y cuánto es sqrt(256)?",
#     "¿Qué es Python según Wikipedia?"
# ]

# for pregunta in preguntas:
#     print(f"\nPregunta: {pregunta}")
#     respuesta = asistente.preguntar(pregunta)
#     print(f"Respuesta: {respuesta}")

# asistente.resumen_uso()
# asistente.exportar_log()

### Extensiones opcionales

1. **Timeout por herramienta:** si `buscar_wikipedia` tarda más de 3 segundos, devuelve un error en lugar de bloquear.

2. **Herramienta nueva:** añade `convertir_divisas` que llame a la API gratuita de `exchangerate-api.com` y registra su uso en el sistema.

3. **Historial persistente:** guarda el historial completo de la conversación en un JSON al finalizar, no solo el log de herramientas.

In [ ]:
# Espacio para las extensiones opcionales

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
*Notebook 02 — Semana 2 del Plan de Formación AI Engineering*